# 6.13 — Weight Decay & Early Stopping

Weight decay and early stopping are two ways to control a model that can fit too much. Weight decay adds an explicit force that pulls parameters toward zero at every update; early stopping adds an implicit force by refusing to keep training after validation performance stops improving. In this notebook we build both ideas from scratch in NumPy, inspect the actual arithmetic, and connect the local update rule to the global behavior of training curves.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build the two regularizers one idea at a time. Run each cell in order and read the printed intermediate values — every update is small enough to inspect, and every plot is meant to show what changed and why.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

▶ What you'll see: NumPy and Matplotlib are loaded, and all random choices are reproducible.

### 1. A plain gradient step can keep large weights large

Start with a single parameter vector. A loss gradient tells us which direction lowers the data loss, so ordinary gradient descent uses `theta - eta * gradient`. That is useful, but it has no built-in reason to make the parameter vector small. If the data gradient is weak or noisy, a large parameter can stay large for a long time.

In [ ]:
theta_w1 = np.array([2.0, -1.0, 0.5])
grad_w1 = np.array([0.3, -0.2, 0.1])
eta_w1 = 0.1
plain_next_w1 = theta_w1 - eta_w1 * grad_w1
print("theta before:", theta_w1)
print("data gradient:", grad_w1)
print("plain next theta:", np.round(plain_next_w1, 3))
assert np.allclose(np.round(plain_next_w1, 3), [1.97, -0.98, 0.49])

▶ What you'll see: the data gradient nudges each coordinate, but the vector is still close to `[2, -1, 0.5]`.

In [ ]:
norm_before_w1 = np.linalg.norm(theta_w1)
norm_after_w1 = np.linalg.norm(plain_next_w1)
print("norm before:", round(norm_before_w1, 3))
print("norm after plain step:", round(norm_after_w1, 3))
plt.figure(figsize=(4.4, 3))
plt.bar(["before", "after plain"], [norm_before_w1, norm_after_w1], color=["gray", "steelblue"])
plt.ylabel("L2 norm ||theta||")
plt.title("1: a plain step barely changes size")
plt.show()
assert round(norm_before_w1, 3) == 2.291

▶ What you'll see: the norm decreases only slightly because the plain update is trying to reduce data loss, not directly control size.

*Why it's done this way:* gradient descent follows the derivative of the data loss. If that derivative is not aligned with the vector itself, the update can reduce error without strongly reducing parameter magnitude. Regularization adds an extra mathematical preference: among many parameter settings that fit the data, prefer smaller ones.

### 2. Weight decay adds multiplicative shrinkage before the gradient step

The lesson's core update is

$$\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta g_t.$$

The new ingredient is the factor `(1 - eta * lambda)`. It multiplies every coordinate before the data-gradient move, so every update has two parts: shrink the old weights, then learn from the current gradient.

In [ ]:
theta_w2 = np.array([2.0, -1.0, 0.5])
grad_w2 = np.array([0.3, -0.2, 0.1])
eta_w2 = 0.1
lam_w2 = 0.2
shrink_w2 = 1 - eta_w2 * lam_w2
shrunk_theta_w2 = shrink_w2 * theta_w2
decay_next_w2 = shrunk_theta_w2 - eta_w2 * grad_w2
print("shrink factor:", round(shrink_w2, 3))
print("after shrink only:", np.round(shrunk_theta_w2, 3))
print("after shrink + gradient:", np.round(decay_next_w2, 3))
assert round(shrink_w2, 3) == 0.98
assert np.allclose(np.round(decay_next_w2, 3), [1.93, -0.96, 0.48])

▶ What you'll see: weight decay moves `[2, -1, 0.5]` to `[1.96, -0.98, 0.49]` before the data gradient is even applied.

In [ ]:
steps_w2 = 25
theta_path_w2 = [theta_w2.copy()]
plain_path_w2 = [theta_w2.copy()]
for step_w2 in range(steps_w2):
    theta_path_w2.append((1 - eta_w2 * lam_w2) * theta_path_w2[-1] - eta_w2 * grad_w2)
    plain_path_w2.append(plain_path_w2[-1] - eta_w2 * grad_w2)
norm_decay_w2 = [np.linalg.norm(v) for v in theta_path_w2]
norm_plain_w2 = [np.linalg.norm(v) for v in plain_path_w2]
print("final plain norm:", round(norm_plain_w2[-1], 3))
print("final decay norm:", round(norm_decay_w2[-1], 3))
plt.figure(figsize=(5, 3))
plt.plot(norm_plain_w2, label="plain GD", color="gray")
plt.plot(norm_decay_w2, label="with weight decay", color="seagreen")
plt.xlabel("step")
plt.ylabel("||theta||")
plt.title("2: repeated shrinkage compounds")
plt.legend()
plt.show()
assert norm_decay_w2[-1] < norm_plain_w2[-1]

▶ What you'll see: the decayed trajectory keeps the parameter norm lower than the plain trajectory at the same learning rate.

*Why it's done this way:* multiplying by `(1 - eta * lambda)` is not a one-time penalty; it compounds over steps. If the data gradient is zero, the parameter becomes `(1 - eta*lambda)^t theta_0`, an exponential decay toward zero. That is why a local shrinkage rule changes global training behavior.

### 3. L2 regularization produces the same shrinkage for ordinary SGD

Weight decay is closely related to adding an L2 penalty to the loss. If the regularized objective is

$$J(\theta)=L(\theta)+\frac{\lambda}{2}\lVert\theta\rVert^2,$$

then the gradient is `data_gradient + lambda * theta`. A gradient descent step becomes `theta - eta * (g + lambda * theta)`, which rearranges to `(1 - eta*lambda)*theta - eta*g`.

In [ ]:
theta_w3 = np.array([2.0, -1.0, 0.5])
grad_data_w3 = np.array([0.3, -0.2, 0.1])
eta_w3 = 0.1
lam_w3 = 0.2
grad_l2_w3 = lam_w3 * theta_w3
grad_total_w3 = grad_data_w3 + grad_l2_w3
l2_step_w3 = theta_w3 - eta_w3 * grad_total_w3
formula_step_w3 = (1 - eta_w3 * lam_w3) * theta_w3 - eta_w3 * grad_data_w3
print("L2 gradient lambda*theta:", grad_l2_w3)
print("total gradient:", grad_total_w3)
print("L2-gradient step:", np.round(l2_step_w3, 3))
print("decay formula step:", np.round(formula_step_w3, 3))
assert np.allclose(l2_step_w3, formula_step_w3)

▶ What you'll see: the L2-gradient update and the weight-decay formula produce exactly the same vector here.

In [ ]:
penalty_before_w3 = 0.5 * lam_w3 * np.sum(theta_w3 ** 2)
penalty_after_w3 = 0.5 * lam_w3 * np.sum(l2_step_w3 ** 2)
print("penalty before:", round(penalty_before_w3, 3))
print("penalty after:", round(penalty_after_w3, 3))
plt.figure(figsize=(4.4, 3))
plt.bar(["before", "after"], [penalty_before_w3, penalty_after_w3], color=["indianred", "seagreen"])
plt.ylabel("lambda/2 * ||theta||^2")
plt.title("3: L2 penalty falls after shrinkage")
plt.show()
assert round(penalty_before_w3, 3) == 0.525

▶ What you'll see: the explicit size penalty falls because the update contains a component pointing back toward the origin.

*Why it's done this way:* differentiating `0.5 * lambda * sum(theta^2)` gives `lambda * theta`, so large coordinates receive proportionally larger pullback. The factor `0.5` is a bookkeeping trick: it cancels the derivative's `2`, leaving the clean gradient `lambda*theta`.

### 4. Early stopping regularizes by limiting training time

Early stopping does not add a term to the update. Instead, it watches validation loss and keeps the parameter setting from the epoch where validation was best. In an over-flexible model, training loss can keep falling while validation loss rises; stopping at the validation minimum chooses a simpler, less memorized solution.

In [ ]:
rng_w4 = np.random.default_rng(0)
x_train_w4 = np.linspace(-1, 1, 8)
y_train_w4 = 1.0 + 2.0 * x_train_w4 + rng_w4.normal(0, 0.35, size=x_train_w4.shape)
x_val_w4 = np.linspace(-0.95, 0.95, 80)
y_val_w4 = 1.0 + 2.0 * x_val_w4
powers_w4 = np.arange(10)
X_train_w4 = x_train_w4[:, None] ** powers_w4[None, :]
X_val_w4 = x_val_w4[:, None] ** powers_w4[None, :]
print("train design shape:", X_train_w4.shape)
print("validation design shape:", X_val_w4.shape)

▶ What you'll see: only 8 training points are being fit with 10 polynomial coefficients, so memorization is plausible.

In [ ]:
theta_w4 = np.zeros(X_train_w4.shape[1])
eta_w4 = 0.02
train_losses_w4 = []
val_losses_w4 = []
norms_w4 = []
best_epoch_w4 = 0
best_val_w4 = float("inf")
best_theta_w4 = theta_w4.copy()
for epoch_w4 in range(1500):
    pred_train_w4 = X_train_w4 @ theta_w4
    grad_w4 = (2 / len(x_train_w4)) * X_train_w4.T @ (pred_train_w4 - y_train_w4)
    theta_w4 = theta_w4 - eta_w4 * grad_w4
    train_loss_w4 = np.mean((X_train_w4 @ theta_w4 - y_train_w4) ** 2)
    val_loss_w4 = np.mean((X_val_w4 @ theta_w4 - y_val_w4) ** 2)
    train_losses_w4.append(train_loss_w4)
    val_losses_w4.append(val_loss_w4)
    norms_w4.append(np.linalg.norm(theta_w4))
    if val_loss_w4 < best_val_w4:
        best_val_w4 = val_loss_w4
        best_epoch_w4 = epoch_w4
        best_theta_w4 = theta_w4.copy()
print("best epoch:", best_epoch_w4)
print("final validation loss:", round(val_losses_w4[-1], 4))
print("best validation loss:", round(best_val_w4, 4))
assert best_epoch_w4 < len(val_losses_w4) - 100
assert best_val_w4 < val_losses_w4[-1]

▶ What you'll see: the best validation epoch occurs well before the final epoch, and the saved validation loss is the one early stopping would keep.

In [ ]:
plt.figure(figsize=(5.2, 3))
plt.plot(train_losses_w4, label="train", color="gray")
plt.plot(val_losses_w4, label="validation", color="darkorange")
plt.axvline(best_epoch_w4, color="seagreen", linestyle="--", label="early-stop epoch")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("4: early stopping chooses the validation minimum")
plt.legend()
plt.show()

▶ What you'll see: the training curve keeps optimizing the training set, while early stopping selects the epoch with the best validation curve.

*Why it's done this way:* validation loss estimates performance on data not used for updates. If the model starts memorizing noise, training loss may improve for the wrong reason. Saving the best validation epoch turns training time into a capacity knob: fewer epochs means fewer chances for high-variance parameters to grow.

### 5. Weight decay and early stopping are complementary capacity controls

Weight decay changes every update; early stopping changes which update we keep. They often work together: decay keeps weights small throughout training, and early stopping prevents the remaining flexibility from chasing validation noise.

In [ ]:
def train_poly_w5(lam_w5, max_epochs_w5=500):
    theta_w5 = np.zeros(X_train_w4.shape[1])
    train_w5, val_w5, norm_w5 = [], [], []
    best_epoch_local_w5 = 0
    best_val_local_w5 = float("inf")
    best_theta_local_w5 = theta_w5.copy()
    for epoch_local_w5 in range(max_epochs_w5):
        pred_w5 = X_train_w4 @ theta_w5
        grad_w5 = (2 / len(x_train_w4)) * X_train_w4.T @ (pred_w5 - y_train_w4)
        theta_w5 = (1 - eta_w4 * lam_w5) * theta_w5 - eta_w4 * grad_w5
        train_loss_local_w5 = np.mean((X_train_w4 @ theta_w5 - y_train_w4) ** 2)
        val_loss_local_w5 = np.mean((X_val_w4 @ theta_w5 - y_val_w4) ** 2)
        train_w5.append(train_loss_local_w5)
        val_w5.append(val_loss_local_w5)
        norm_w5.append(np.linalg.norm(theta_w5))
        if val_loss_local_w5 < best_val_local_w5:
            best_val_local_w5 = val_loss_local_w5
            best_epoch_local_w5 = epoch_local_w5
            best_theta_local_w5 = theta_w5.copy()
    return np.array(train_w5), np.array(val_w5), np.array(norm_w5), best_epoch_local_w5, best_theta_local_w5

train0_w5, val0_w5, norm0_w5, best0_w5, theta0_w5 = train_poly_w5(0.0)
train_decay_w5, val_decay_w5, norm_decay_w5, best_decay_w5, theta_decay_w5 = train_poly_w5(0.15)
print("best val no decay:", round(float(np.min(val0_w5)), 4), "epoch", best0_w5)
print("best val with decay:", round(float(np.min(val_decay_w5)), 4), "epoch", best_decay_w5)
print("final norm no decay:", round(norm0_w5[-1], 3))
print("final norm with decay:", round(norm_decay_w5[-1], 3))
assert norm_decay_w5[-1] < norm0_w5[-1]

▶ What you'll see: decay keeps the coefficient norm smaller, while each run still has a best validation epoch.

In [ ]:
plt.figure(figsize=(5.2, 3))
plt.plot(val0_w5, label="validation, no decay", color="gray")
plt.plot(val_decay_w5, label="validation, decay", color="seagreen")
plt.axvline(best0_w5, color="gray", linestyle="--")
plt.axvline(best_decay_w5, color="seagreen", linestyle="--")
plt.xlabel("epoch")
plt.ylabel("validation MSE")
plt.title("5: explicit shrinkage + choosing when to stop")
plt.legend()
plt.show()

▶ What you'll see: validation curves can improve differently under decay, and early stopping chooses the best point on each curve.

*Why it's done this way:* weight decay is an explicit prior toward small parameters; early stopping is an implicit prior toward parameters reachable in limited time. Both reduce effective capacity, but they act at different places in the training loop, so using both is common rather than redundant.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

▶ What you'll see: the only libraries used in the lesson are loaded.

## 🟢 Basics (warm-up)

### Basic 1 — Measure parameter size with an L2 norm

**Goal.** Compute a weight vector's size, because weight decay is a rule for controlling that size.

In [ ]:
theta_b1 = np.array([3.0, 4.0])
norm_b1 = np.linalg.norm(theta_b1)
penalty_b1 = np.sum(theta_b1 ** 2)
print("theta:", theta_b1)
print("L2 norm:", norm_b1)
print("squared norm:", penalty_b1)
plt.figure(figsize=(4, 3))
plt.bar(["theta0²", "theta1²"], theta_b1 ** 2, color="steelblue")
plt.title("Basic 1: squared coordinates behind ||theta||")
plt.ylabel("squared value")
plt.show()
assert norm_b1 == 5.0
assert penalty_b1 == 25.0

▶ What you'll see: `[3,4]` has norm 5, and the larger coordinate contributes more squared penalty.

👀 Takeaway: L2 regularization charges parameters by squared size, so large coordinates become expensive.

### Basic 2 — Compute one shrinkage factor

**Goal.** Calculate `(1 - eta * lambda)`, because this is the multiplicative part of weight decay.

In [ ]:
eta_b2 = 0.1
lam_b2 = 0.3
shrink_b2 = 1 - eta_b2 * lam_b2
theta_b2 = np.array([2.0, -2.0])
shrunk_b2 = shrink_b2 * theta_b2
print("shrink factor:", round(shrink_b2, 3))
print("before:", theta_b2)
print("after shrink only:", np.round(shrunk_b2, 3))
plt.figure(figsize=(4, 3))
plt.bar(["before norm", "after norm"], [np.linalg.norm(theta_b2), np.linalg.norm(shrunk_b2)], color=["gray", "seagreen"])
plt.title("Basic 2: shrink factor reduces norm")
plt.ylabel("L2 norm")
plt.show()
assert round(shrink_b2, 3) == 0.97

▶ What you'll see: every coordinate is multiplied by 0.97 before considering the data gradient.

👀 Takeaway: weight decay is a repeated proportional pull toward zero.

### Basic 3 — Compare a plain step with a decayed step

**Goal.** Put the same gradient into two update rules, because the only difference should be the shrinkage term.

In [ ]:
theta_b3 = np.array([1.5, -0.5])
grad_b3 = np.array([0.2, 0.4])
eta_b3 = 0.2
lam_b3 = 0.1
plain_b3 = theta_b3 - eta_b3 * grad_b3
decay_b3 = (1 - eta_b3 * lam_b3) * theta_b3 - eta_b3 * grad_b3
print("plain step:", np.round(plain_b3, 3))
print("decayed step:", np.round(decay_b3, 3))
plt.figure(figsize=(4, 3))
plt.scatter([plain_b3[0], decay_b3[0]], [plain_b3[1], decay_b3[1]], s=90, color=["gray", "seagreen"])
plt.text(plain_b3[0], plain_b3[1], "plain")
plt.text(decay_b3[0], decay_b3[1], "decay")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.title("Basic 3: two next parameters")
plt.show()
assert np.linalg.norm(decay_b3) < np.linalg.norm(plain_b3)

▶ What you'll see: the decayed update lands closer to the origin than the plain update.

👀 Takeaway: weight decay changes the parameter update even when the data gradient is identical.

### Basic 4 — Repeated decay with zero data gradient

**Goal.** Isolate shrinkage by setting the data gradient to zero, because this shows what decay does on its own.

In [ ]:
theta_b4 = np.array([2.0])
eta_b4 = 0.1
lam_b4 = 0.5
values_b4 = []
for step_b4 in range(8):
    values_b4.append(float(theta_b4[0]))
    theta_b4 = (1 - eta_b4 * lam_b4) * theta_b4
print("trajectory:", np.round(values_b4, 3))
plt.figure(figsize=(4, 3))
plt.plot(values_b4, marker="o", color="purple")
plt.title("Basic 4: exponential shrinkage")
plt.xlabel("step")
plt.ylabel("theta")
plt.show()
assert round(values_b4[1], 3) == 1.9

▶ What you'll see: the parameter follows `2, 1.9, 1.805, ...` because it is multiplied by 0.95 each step.

👀 Takeaway: with no data force, weight decay drives parameters exponentially toward zero.

### Basic 5 — Compute the L2 penalty term

**Goal.** Evaluate `lambda/2 * ||theta||²`, because this is the objective term whose gradient creates shrinkage.

In [ ]:
theta_b5 = np.array([2.0, -1.0, 0.5])
lam_b5 = 0.2
sq_norm_b5 = np.sum(theta_b5 ** 2)
reg_b5 = 0.5 * lam_b5 * sq_norm_b5
print("squared norm:", round(sq_norm_b5, 3))
print("L2 penalty:", round(reg_b5, 3))
plt.figure(figsize=(4, 3))
plt.bar(["||theta||²", "lambda/2 * ||theta||²"], [sq_norm_b5, reg_b5], color=["orange", "teal"])
plt.title("Basic 5: raw size vs scaled penalty")
plt.xticks(rotation=12)
plt.show()
assert round(reg_b5, 3) == 0.525

▶ What you'll see: the regularization term is a scaled version of the squared norm.

👀 Takeaway: λ controls how much the optimizer cares about weight size relative to data fit.

### Basic 6 — One linear-regression update with decay

**Goal.** Apply weight decay in a tiny regression problem, because real training combines data gradients and shrinkage.

In [ ]:
X_b6 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
y_b6 = np.array([1.0, 2.0, 3.0])
theta_b6 = np.array([0.5, 0.5])
eta_b6 = 0.1
lam_b6 = 0.2
pred_b6 = X_b6 @ theta_b6
grad_b6 = (2 / len(y_b6)) * X_b6.T @ (pred_b6 - y_b6)
next_b6 = (1 - eta_b6 * lam_b6) * theta_b6 - eta_b6 * grad_b6
print("prediction before:", np.round(pred_b6, 3))
print("data gradient:", np.round(grad_b6, 3))
print("theta after decay step:", np.round(next_b6, 3))
plt.figure(figsize=(4, 3))
plt.plot([0, 1, 2], y_b6, "o", label="data")
plt.plot([0, 1, 2], X_b6 @ next_b6, label="after one step")
plt.title("Basic 6: one decayed regression step")
plt.legend()
plt.show()
assert np.allclose(np.round(next_b6, 3), [0.69, 0.757])

▶ What you'll see: the line moves toward the data while the update still includes shrinkage.

👀 Takeaway: weight decay does not replace the data gradient; it adds a size-control force to it.

### Basic 7 — Make a train/validation split

**Goal.** Separate data used for updates from data used for stopping, because early stopping needs independent feedback.

In [ ]:
x_b7 = np.linspace(-1, 1, 20)
y_b7 = 1.0 + 2.0 * x_b7
train_idx_b7 = np.arange(0, 20, 2)
val_idx_b7 = np.arange(1, 20, 2)
x_train_b7, y_train_b7 = x_b7[train_idx_b7], y_b7[train_idx_b7]
x_val_b7, y_val_b7 = x_b7[val_idx_b7], y_b7[val_idx_b7]
print("train size:", len(x_train_b7))
print("validation size:", len(x_val_b7))
plt.figure(figsize=(4, 3))
plt.scatter(x_train_b7, y_train_b7, label="train", color="steelblue")
plt.scatter(x_val_b7, y_val_b7, label="validation", color="darkorange")
plt.title("Basic 7: split before stopping")
plt.legend()
plt.show()
assert len(x_train_b7) == 10 and len(x_val_b7) == 10

▶ What you'll see: alternating points are assigned to training and validation.

👀 Takeaway: early stopping only means something if validation loss is not used to compute gradients.

### Basic 8 — Track train and validation loss

**Goal.** Record both losses during training, because early stopping is a decision made from a curve, not a single update.

In [ ]:
X_train_b8 = np.c_[np.ones_like(x_train_b7), x_train_b7]
X_val_b8 = np.c_[np.ones_like(x_val_b7), x_val_b7]
theta_b8 = np.zeros(2)
eta_b8 = 0.2
train_losses_b8 = []
val_losses_b8 = []
for epoch_b8 in range(25):
    grad_b8 = (2 / len(y_train_b7)) * X_train_b8.T @ (X_train_b8 @ theta_b8 - y_train_b7)
    theta_b8 = theta_b8 - eta_b8 * grad_b8
    train_losses_b8.append(np.mean((X_train_b8 @ theta_b8 - y_train_b7) ** 2))
    val_losses_b8.append(np.mean((X_val_b8 @ theta_b8 - y_val_b7) ** 2))
print("first val loss:", round(val_losses_b8[0], 3))
print("last val loss:", round(val_losses_b8[-1], 6))
plt.figure(figsize=(4.5, 3))
plt.plot(train_losses_b8, label="train")
plt.plot(val_losses_b8, label="validation")
plt.title("Basic 8: monitored losses")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()
assert val_losses_b8[-1] < val_losses_b8[0]

▶ What you'll see: both losses fall on this simple well-specified line problem.

👀 Takeaway: monitoring validation loss is the raw material for an early-stopping rule.

### Basic 9 — Implement a patience counter

**Goal.** Stop after several epochs without improvement, because validation curves can wiggle rather than improve monotonically.

In [ ]:
val_curve_b9 = np.array([0.90, 0.70, 0.62, 0.61, 0.615, 0.618, 0.620, 0.590])
patience_b9 = 2
best_b9 = float("inf")
best_epoch_b9 = -1
wait_b9 = 0
stop_epoch_b9 = None
for epoch_b9, loss_b9 in enumerate(val_curve_b9):
    if loss_b9 < best_b9:
        best_b9 = loss_b9
        best_epoch_b9 = epoch_b9
        wait_b9 = 0
    else:
        wait_b9 += 1
        if wait_b9 >= patience_b9:
            stop_epoch_b9 = epoch_b9
            break
print("best epoch before stop:", best_epoch_b9)
print("stop epoch:", stop_epoch_b9)
plt.figure(figsize=(4, 3))
plt.plot(val_curve_b9, marker="o", color="darkorange")
plt.axvline(best_epoch_b9, color="seagreen", linestyle="--", label="best kept")
plt.axvline(stop_epoch_b9, color="red", linestyle=":", label="stop")
plt.title("Basic 9: patience rule")
plt.legend()
plt.show()
assert best_epoch_b9 == 3 and stop_epoch_b9 == 5

▶ What you'll see: the rule keeps epoch 3 and stops at epoch 5 after two non-improving epochs.

👀 Takeaway: patience protects against stopping on the first tiny upward wiggle.

### Basic 10 — Plot parameter norm across epochs

**Goal.** Watch weight size during training, because regularization is about controlling both loss and parameter growth.

In [ ]:
X_b10 = X_train_b8
y_b10 = y_train_b7
theta_b10 = np.zeros(2)
eta_b10 = 0.2
lam_b10 = 0.1
norms_b10 = []
losses_b10 = []
for epoch_b10 in range(30):
    grad_b10 = (2 / len(y_b10)) * X_b10.T @ (X_b10 @ theta_b10 - y_b10)
    theta_b10 = (1 - eta_b10 * lam_b10) * theta_b10 - eta_b10 * grad_b10
    norms_b10.append(np.linalg.norm(theta_b10))
    losses_b10.append(np.mean((X_b10 @ theta_b10 - y_b10) ** 2))
print("final theta:", np.round(theta_b10, 3))
print("final norm:", round(norms_b10[-1], 3))
plt.figure(figsize=(4.5, 3))
plt.plot(norms_b10, label="||theta||", color="seagreen")
plt.plot(losses_b10, label="train MSE", color="gray")
plt.title("Basic 10: loss and norm are different signals")
plt.legend()
plt.show()
assert norms_b10[-1] > 0

▶ What you'll see: the model learns while the norm stays explicitly controlled by decay.

👀 Takeaway: weight decay is easiest to debug by plotting both objective behavior and parameter size.

## 🟡 Easy

### Easy 1 — Train linear regression with and without weight decay

**Goal.** Compare two models on the same data, because decay should trade a little training flexibility for smaller weights.

In [ ]:
rng_e1 = np.random.default_rng(1)
x_e1 = np.linspace(-1, 1, 30)
y_e1 = 0.5 + 2.0 * x_e1 + rng_e1.normal(0, 0.08, size=x_e1.shape)
X_e1 = np.c_[np.ones_like(x_e1), x_e1]
def fit_decay_e1(lam_e1):
    theta_e1 = np.zeros(2)
    losses_e1 = []
    for epoch_e1 in range(120):
        grad_e1 = (2 / len(y_e1)) * X_e1.T @ (X_e1 @ theta_e1 - y_e1)
        theta_e1 = (1 - 0.15 * lam_e1) * theta_e1 - 0.15 * grad_e1
        losses_e1.append(np.mean((X_e1 @ theta_e1 - y_e1) ** 2))
    return theta_e1, np.array(losses_e1)
theta0_e1, loss0_e1 = fit_decay_e1(0.0)
thetad_e1, lossd_e1 = fit_decay_e1(0.2)
print("no decay theta:", np.round(theta0_e1, 3))
print("decay theta:", np.round(thetad_e1, 3))
print("norms:", round(np.linalg.norm(theta0_e1), 3), round(np.linalg.norm(thetad_e1), 3))
plt.figure(figsize=(5, 3))
plt.plot(loss0_e1, label="no decay", color="gray")
plt.plot(lossd_e1, label="decay", color="seagreen")
plt.title("Easy 1: training loss comparison")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()
assert np.linalg.norm(thetad_e1) < np.linalg.norm(theta0_e1)

▶ What you'll see: both runs fit the line, but the decayed model has the smaller parameter norm.

👀 Takeaway: decay is a capacity-control knob, not a guarantee that training loss will be lowest.

### Easy 2 — Sweep λ and compare train/validation error

**Goal.** Tune weight decay with validation data, because too much shrinkage can underfit.

In [ ]:
rng_e2 = np.random.default_rng(2)
x_all_e2 = np.linspace(-1, 1, 50)
y_true_e2 = 1.0 - 1.5 * x_all_e2 + 0.8 * x_all_e2 ** 2
y_all_e2 = y_true_e2 + rng_e2.normal(0, 0.08, size=x_all_e2.shape)
train_mask_e2 = np.arange(len(x_all_e2)) % 2 == 0
X_all_e2 = np.c_[np.ones_like(x_all_e2), x_all_e2, x_all_e2 ** 2, x_all_e2 ** 3]
X_train_e2, y_train_e2 = X_all_e2[train_mask_e2], y_all_e2[train_mask_e2]
X_val_e2, y_val_e2 = X_all_e2[~train_mask_e2], y_true_e2[~train_mask_e2]
lams_e2 = np.array([0.0, 0.01, 0.05, 0.2, 0.8])
train_rmse_e2 = []
val_rmse_e2 = []
for lam_e2 in lams_e2:
    theta_e2 = np.zeros(X_train_e2.shape[1])
    for epoch_e2 in range(250):
        grad_e2 = (2 / len(y_train_e2)) * X_train_e2.T @ (X_train_e2 @ theta_e2 - y_train_e2)
        theta_e2 = (1 - 0.08 * lam_e2) * theta_e2 - 0.08 * grad_e2
    train_rmse_e2.append(np.sqrt(np.mean((X_train_e2 @ theta_e2 - y_train_e2) ** 2)))
    val_rmse_e2.append(np.sqrt(np.mean((X_val_e2 @ theta_e2 - y_val_e2) ** 2)))
print("validation RMSE:", np.round(val_rmse_e2, 3))
plt.figure(figsize=(5, 3))
plt.plot(lams_e2, train_rmse_e2, marker="o", label="train")
plt.plot(lams_e2, val_rmse_e2, marker="o", label="validation")
plt.title("Easy 2: λ sweep")
plt.xlabel("lambda")
plt.ylabel("RMSE")
plt.legend()
plt.show()
assert len(val_rmse_e2) == 5

▶ What you'll see: validation error changes across λ values, so λ should be selected rather than guessed.

👀 Takeaway: tune decay strength on validation performance, not on the desire for smaller weights alone.

### Easy 3 — Choose an early-stopping epoch from a validation curve

**Goal.** Save the best model state, because the last epoch is not always the best epoch.

In [ ]:
rng_e3 = np.random.default_rng(3)
x_train_e3 = np.linspace(-1, 1, 14)
y_train_e3 = 0.4 + 1.8 * x_train_e3 + rng_e3.normal(0, 0.18, size=x_train_e3.shape)
x_val_e3 = np.linspace(-0.9, 0.9, 60)
y_val_e3 = 0.4 + 1.8 * x_val_e3
powers_e3 = np.arange(9)
X_train_e3 = x_train_e3[:, None] ** powers_e3[None, :]
X_val_e3 = x_val_e3[:, None] ** powers_e3[None, :]
theta_e3 = np.zeros(len(powers_e3))
val_losses_e3 = []
best_loss_e3 = float("inf")
best_epoch_e3 = 0
best_theta_e3 = theta_e3.copy()
for epoch_e3 in range(350):
    grad_e3 = (2 / len(y_train_e3)) * X_train_e3.T @ (X_train_e3 @ theta_e3 - y_train_e3)
    theta_e3 = theta_e3 - 0.035 * grad_e3
    val_loss_e3 = np.mean((X_val_e3 @ theta_e3 - y_val_e3) ** 2)
    val_losses_e3.append(val_loss_e3)
    if val_loss_e3 < best_loss_e3:
        best_loss_e3 = val_loss_e3
        best_epoch_e3 = epoch_e3
        best_theta_e3 = theta_e3.copy()
print("best epoch:", best_epoch_e3)
print("best vs final val:", round(best_loss_e3, 4), round(val_losses_e3[-1], 4))
plt.figure(figsize=(5, 3))
plt.plot(val_losses_e3, color="darkorange")
plt.axvline(best_epoch_e3, color="seagreen", linestyle="--")
plt.title("Easy 3: saved best validation epoch")
plt.xlabel("epoch")
plt.ylabel("validation MSE")
plt.show()
assert best_loss_e3 <= val_losses_e3[-1]

▶ What you'll see: the stored best model is selected by the minimum validation loss.

👀 Takeaway: early stopping is model selection over time.

### Easy 4 — Combine decay and early stopping

**Goal.** Use both regularizers in one loop, because modern training often shrinks weights and monitors validation.

In [ ]:
lam_e4 = 0.1
theta_e4 = np.zeros(len(powers_e3))
train_losses_e4 = []
val_losses_e4 = []
norms_e4 = []
best_val_e4 = float("inf")
best_epoch_e4 = 0
for epoch_e4 in range(350):
    grad_e4 = (2 / len(y_train_e3)) * X_train_e3.T @ (X_train_e3 @ theta_e4 - y_train_e3)
    theta_e4 = (1 - 0.035 * lam_e4) * theta_e4 - 0.035 * grad_e4
    train_losses_e4.append(np.mean((X_train_e3 @ theta_e4 - y_train_e3) ** 2))
    val_losses_e4.append(np.mean((X_val_e3 @ theta_e4 - y_val_e3) ** 2))
    norms_e4.append(np.linalg.norm(theta_e4))
    if val_losses_e4[-1] < best_val_e4:
        best_val_e4 = val_losses_e4[-1]
        best_epoch_e4 = epoch_e4
print("best epoch with decay:", best_epoch_e4)
print("final norm:", round(norms_e4[-1], 3))
plt.figure(figsize=(5, 3))
plt.plot(val_losses_e4, label="validation", color="seagreen")
plt.plot(np.array(norms_e4) / max(norms_e4), label="scaled norm", color="gray")
plt.axvline(best_epoch_e4, linestyle="--", color="black")
plt.title("Easy 4: decay plus early stopping")
plt.legend()
plt.show()
assert best_val_e4 <= val_losses_e4[-1]

▶ What you'll see: validation loss and parameter norm are monitored in the same training run.

👀 Takeaway: decay controls every update, while early stopping controls which epoch survives.

### Easy 5 — Standardize features before comparing decay strengths

**Goal.** Show why scale matters, because L2 penalties are applied to coefficients and coefficients depend on feature units.

In [ ]:
rng_e5 = np.random.default_rng(5)
x_raw_e5 = np.linspace(0, 100, 40)
y_e5 = 3.0 + 0.04 * x_raw_e5 + rng_e5.normal(0, 0.2, size=x_raw_e5.shape)
x_std_e5 = (x_raw_e5 - np.mean(x_raw_e5)) / np.std(x_raw_e5)
X_raw_e5 = np.c_[np.ones_like(x_raw_e5), x_raw_e5]
X_std_e5 = np.c_[np.ones_like(x_std_e5), x_std_e5]
def fit_scaled_e5(X_e5, eta_local_e5):
    theta_e5 = np.zeros(X_e5.shape[1])
    for epoch_e5 in range(300):
        grad_e5 = (2 / len(y_e5)) * X_e5.T @ (X_e5 @ theta_e5 - y_e5)
        theta_e5 = (1 - eta_local_e5 * 0.1) * theta_e5 - eta_local_e5 * grad_e5
    return theta_e5
theta_raw_e5 = fit_scaled_e5(X_raw_e5, 0.000001)
theta_std_e5 = fit_scaled_e5(X_std_e5, 0.05)
print("raw-scale theta:", np.round(theta_raw_e5, 3))
print("standardized theta:", np.round(theta_std_e5, 3))
plt.figure(figsize=(4, 3))
plt.bar(["raw norm", "standardized norm"], [np.linalg.norm(theta_raw_e5), np.linalg.norm(theta_std_e5)], color=["red", "teal"])
plt.title("Easy 5: feature scale changes coefficient size")
plt.ylabel("||theta||")
plt.show()
assert np.isfinite(theta_raw_e5).all() and np.isfinite(theta_std_e5).all()

▶ What you'll see: coefficient magnitudes depend strongly on feature scaling.

👀 Takeaway: because decay penalizes coefficients, feature scale changes what the same λ means.

## 🔴 Advanced

### Advanced 1 — View early stopping as a spectral filter

**Goal.** Show that gradient descent learns high-curvature directions faster, because stopping early suppresses slow directions.

In [ ]:
singular_values_a1 = np.array([3.0, 1.0, 0.25])
eta_a1 = 0.08
steps_a1 = np.arange(0, 80)
filters_a1 = np.array([1 - (1 - eta_a1 * s_a1 ** 2) ** steps_a1 for s_a1 in singular_values_a1])
print("filter at step 10:", np.round(filters_a1[:, 10], 3))
print("filter at step 60:", np.round(filters_a1[:, 60], 3))
plt.figure(figsize=(5, 3))
for idx_a1, s_a1 in enumerate(singular_values_a1):
    plt.plot(steps_a1, filters_a1[idx_a1], label=f"singular value {s_a1}")
plt.title("Advanced 1: early stopping filters slow directions")
plt.xlabel("gradient steps")
plt.ylabel("learned fraction")
plt.legend()
plt.show()
assert filters_a1[0, 10] > filters_a1[-1, 10]

▶ What you'll see: large singular directions are learned quickly, while low-signal directions remain suppressed early on.

👀 Takeaway: early stopping regularizes because limited time prevents slow, often noisy directions from fully entering the model.

### Advanced 2 — Compare coupled L2 and decoupled weight decay under preconditioning

**Goal.** Show a subtle optimizer issue, because L2-as-gradient and decoupled decay are identical for plain SGD but not after coordinate-wise scaling.

In [ ]:
theta_a2 = np.array([2.0, -1.0])
grad_data_a2 = np.array([0.4, -0.2])
precond_a2 = np.array([0.1, 2.0])
eta_a2 = 0.1
lam_a2 = 0.2
coupled_a2 = theta_a2 - eta_a2 * precond_a2 * (grad_data_a2 + lam_a2 * theta_a2)
decoupled_a2 = (1 - eta_a2 * lam_a2) * theta_a2 - eta_a2 * precond_a2 * grad_data_a2
print("coupled L2 step:", np.round(coupled_a2, 3))
print("decoupled decay step:", np.round(decoupled_a2, 3))
plt.figure(figsize=(4, 3))
plt.scatter(coupled_a2[0], coupled_a2[1], s=90, label="coupled L2", color="red")
plt.scatter(decoupled_a2[0], decoupled_a2[1], s=90, label="decoupled decay", color="teal")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.title("Advanced 2: preconditioning breaks equivalence")
plt.legend()
plt.show()
assert not np.allclose(coupled_a2, decoupled_a2)

▶ What you'll see: the two updates land in different places because the preconditioner scales the L2 gradient unevenly.

👀 Takeaway: in adaptive optimizers, decoupled weight decay preserves uniform shrinkage more directly than adding λθ to the gradient.

### Advanced 3 — Add min_delta to patience

**Goal.** Ignore tiny validation improvements, because numerical noise can reset patience without meaningful generalization gains.

In [ ]:
val_curve_a3 = np.array([1.00, 0.82, 0.790, 0.786, 0.784, 0.783, 0.782, 0.781])
min_delta_a3 = 0.005
patience_a3 = 2
best_a3 = float("inf")
wait_a3 = 0
stop_a3 = None
accepted_a3 = []
for epoch_a3, loss_a3 in enumerate(val_curve_a3):
    improved_a3 = loss_a3 < best_a3 - min_delta_a3
    accepted_a3.append(improved_a3)
    if improved_a3:
        best_a3 = loss_a3
        wait_a3 = 0
    else:
        wait_a3 += 1
        if wait_a3 >= patience_a3:
            stop_a3 = epoch_a3
            break
print("accepted improvements:", accepted_a3)
print("stop epoch:", stop_a3)
plt.figure(figsize=(4.5, 3))
plt.plot(val_curve_a3, marker="o", color="darkorange")
plt.axvline(stop_a3, color="red", linestyle="--")
plt.title("Advanced 3: min_delta patience")
plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.show()
assert stop_a3 == 6

▶ What you'll see: tiny decreases smaller than `min_delta` no longer count as real improvements.

👀 Takeaway: `min_delta` makes early stopping a practical rule rather than a reaction to noise.

### Advanced 4 — Smooth a noisy validation curve before stopping

**Goal.** Use a moving average for decisions, because validation estimates from small batches can be noisy.

In [ ]:
rng_a4 = np.random.default_rng(44)
base_a4 = 0.5 + 0.5 * np.exp(-np.linspace(0, 4, 40))
noise_a4 = rng_a4.normal(0, 0.015, size=base_a4.shape)
val_noisy_a4 = base_a4 + noise_a4
window_a4 = 5
smooth_a4 = np.convolve(val_noisy_a4, np.ones(window_a4) / window_a4, mode="valid")
best_raw_a4 = int(np.argmin(val_noisy_a4))
best_smooth_a4 = int(np.argmin(smooth_a4) + window_a4 - 1)
print("best raw epoch:", best_raw_a4)
print("best smoothed epoch:", best_smooth_a4)
plt.figure(figsize=(5, 3))
plt.plot(val_noisy_a4, label="raw validation", color="lightgray")
plt.plot(np.arange(window_a4 - 1, len(val_noisy_a4)), smooth_a4, label="moving average", color="teal")
plt.axvline(best_smooth_a4, color="seagreen", linestyle="--")
plt.title("Advanced 4: smoothing validation noise")
plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.legend()
plt.show()
assert 0 <= best_smooth_a4 < len(val_noisy_a4)

▶ What you'll see: the smoothed curve removes small wiggles and gives a steadier stopping signal.

👀 Takeaway: smoothing can make early stopping less sensitive to random validation fluctuations.

### Advanced 5 — Keep a final test set untouched

**Goal.** Separate validation selection from final evaluation, because choosing λ and stop epoch on validation makes validation optimistic.

In [ ]:
rng_a5 = np.random.default_rng(55)
x_a5 = np.linspace(-1, 1, 90)
y_true_a5 = 0.7 + 1.2 * x_a5 - 0.5 * x_a5 ** 2
y_a5 = y_true_a5 + rng_a5.normal(0, 0.1, size=x_a5.shape)
train_a5 = np.arange(0, 90, 3)
val_a5 = np.arange(1, 90, 3)
test_a5 = np.arange(2, 90, 3)
X_a5 = np.c_[np.ones_like(x_a5), x_a5, x_a5 ** 2, x_a5 ** 3, x_a5 ** 4]
lams_a5 = np.array([0.0, 0.03, 0.1, 0.3])
val_scores_a5 = []
test_scores_a5 = []
for lam_a5 in lams_a5:
    theta_a5 = np.zeros(X_a5.shape[1])
    best_val_loss_a5 = float("inf")
    best_theta_a5 = theta_a5.copy()
    for epoch_a5 in range(300):
        grad_a5 = (2 / len(train_a5)) * X_a5[train_a5].T @ (X_a5[train_a5] @ theta_a5 - y_a5[train_a5])
        theta_a5 = (1 - 0.06 * lam_a5) * theta_a5 - 0.06 * grad_a5
        val_loss_a5 = np.mean((X_a5[val_a5] @ theta_a5 - y_a5[val_a5]) ** 2)
        if val_loss_a5 < best_val_loss_a5:
            best_val_loss_a5 = val_loss_a5
            best_theta_a5 = theta_a5.copy()
    val_scores_a5.append(np.sqrt(best_val_loss_a5))
    test_scores_a5.append(np.sqrt(np.mean((X_a5[test_a5] @ best_theta_a5 - y_a5[test_a5]) ** 2)))
best_idx_a5 = int(np.argmin(val_scores_a5))
print("validation RMSE by lambda:", np.round(val_scores_a5, 3))
print("test RMSE by lambda:", np.round(test_scores_a5, 3))
print("chosen lambda:", float(lams_a5[best_idx_a5]))
plt.figure(figsize=(5, 3))
plt.plot(lams_a5, val_scores_a5, marker="o", label="validation")
plt.plot(lams_a5, test_scores_a5, marker="o", label="test")
plt.axvline(lams_a5[best_idx_a5], color="black", linestyle="--", label="chosen on val")
plt.title("Advanced 5: validation chooses, test reports")
plt.xlabel("lambda")
plt.ylabel("RMSE")
plt.legend()
plt.show()
assert len(val_scores_a5) == len(lams_a5)

▶ What you'll see: λ is selected by validation RMSE, while test RMSE is reported only after that choice is fixed.

👀 Takeaway: validation is for decisions; the test set is for the final unbiased estimate.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Weight decay shrinks parameters directly; early stopping shrinks them indirectly by limiting training time.

Both methods control capacity without changing the architecture. Decay changes every step; early stopping chooses the best validation epoch. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def split_scale(X, y):
    if len(y) > 300:
        x_small, _, y_small, _ = train_test_split(X, y, train_size=300, random_state=6, stratify=y)
    else:
        x_small = X
        y_small = y
    x_tr, x_te, y_tr, y_te = train_test_split(x_small, y_small, test_size=0.4, random_state=0, stratify=y_small)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=1, keepdims=True)


def relu(z):
    return np.maximum(z, 0.0)


def init_weights(n_in, n_hidden, n_out, mode, seed):
    rng = np.random.default_rng(seed)
    if mode == "xavier":
        scale1 = math.sqrt(2.0 / (n_in + n_hidden))
        scale2 = math.sqrt(2.0 / (n_hidden + n_out))
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "he":
        scale1 = math.sqrt(2.0 / n_in)
        scale2 = math.sqrt(2.0 / n_hidden)
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "tiny":
        W1 = rng.normal(0.0, 0.01, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.01, size=(n_hidden, n_out))
    elif mode == "large":
        W1 = rng.normal(0.0, 2.0, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 2.0, size=(n_hidden, n_out))
    elif mode == "orthogonal":
        Q1, _ = np.linalg.qr(rng.normal(size=(n_in, max(n_in, n_hidden))))
        Q2, _ = np.linalg.qr(rng.normal(size=(n_hidden, max(n_hidden, n_out))))
        W1 = Q1[:, :n_hidden]
        W2 = Q2[:, :n_out]
    else:
        W1 = rng.normal(0.0, 0.1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.1, size=(n_hidden, n_out))
    b1 = np.zeros(n_hidden)
    b2 = np.zeros(n_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(params, X, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    W1 = params["W1"]
    if dropconnect_p > 0.0 and rng is not None:
        keep_w = 1.0 - dropconnect_p
        mask_w = rng.binomial(1, keep_w, size=W1.shape) / keep_w
        W1 = W1 * mask_w
    z1 = X @ W1 + params["b1"]
    h1 = relu(z1)
    mask = None
    if dropout_p > 0.0 and rng is not None:
        keep = 1.0 - dropout_p
        mask = rng.binomial(1, keep, size=h1.shape) / keep
        h1 = h1 * mask
    logits = h1 @ params["W2"] + params["b2"]
    return z1, h1, logits, mask


def loss_and_grads(params, X, y, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    classes = params["b2"].shape[0]
    z1, h1, logits, mask = forward(params, X, dropout_p, rng, dropconnect_p)
    probs = softmax(logits)
    target = one_hot(y, classes)
    loss = -np.mean(np.sum(target * np.log(probs + 1e-12), axis=1))
    dlogits = (probs - target) / len(y)
    dW2 = h1.T @ dlogits
    db2 = np.sum(dlogits, axis=0)
    dh1 = dlogits @ params["W2"].T
    if mask is not None:
        dh1 = dh1 * mask
    dz1 = dh1 * (z1 > 0.0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads


def predict(params, X):
    _, _, logits, _ = forward(params, X)
    return np.argmax(logits, axis=1)


def eval_loss(params, X, y):
    classes = params["b2"].shape[0]
    _, _, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    return -float(np.mean(np.sum(target * np.log(probs + 1e-12), axis=1)))


def vector_norm(params):
    total = 0.0
    for value in params.values():
        total += float(np.sum(value * value))
    return math.sqrt(total)


def kfac_precondition_grads(params, X, y, grads, damping):
    classes = params["b2"].shape[0]
    z1, h1, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    dlogits = (probs - target) / len(y)
    dh1 = dlogits @ params["W2"].T
    dz1 = dh1 * (z1 > 0.0)
    out = {key: value.copy() for key, value in grads.items()}
    A1 = X.T @ X / len(y) + damping * np.eye(X.shape[1])
    S1 = dz1.T @ dz1 / len(y) + damping * np.eye(dz1.shape[1])
    A2 = h1.T @ h1 / len(y) + damping * np.eye(h1.shape[1])
    S2 = dlogits.T @ dlogits / len(y) + damping * np.eye(dlogits.shape[1])
    out["W1"] = np.linalg.solve(A1, grads["W1"]) @ np.linalg.inv(S1)
    out["W2"] = np.linalg.solve(A2, grads["W2"]) @ np.linalg.inv(S2)
    return out


def apply_update(params, grads, state, method, lr, t, config):
    beta1 = config.get("beta1", 0.9)
    beta2 = config.get("beta2", 0.999)
    eps = config.get("eps", 1e-8)
    mu = config.get("momentum", 0.0)
    weight_decay = config.get("weight_decay", 0.0)
    for key in params:
        grad = grads[key]
        if method == "sgd":
            update = -lr * grad
        elif method == "momentum":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        elif method == "adagrad":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc += grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "rmsprop":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc *= beta2
            acc += (1.0 - beta2) * grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "adam" or method == "adamw":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            update = -lr * m_hat / (np.sqrt(v_hat) + eps)
            if method == "adamw" and key.startswith("W"):
                update -= lr * weight_decay * params[key]
        elif method == "lion":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            blended = beta1 * m + (1.0 - beta1) * grad
            update = -lr * np.sign(blended)
            m *= beta2
            m += (1.0 - beta2) * grad
        elif method == "lamb":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            raw = m / (np.sqrt(v) + eps)
            if key.startswith("W"):
                raw += weight_decay * params[key]
            ratio = np.linalg.norm(params[key]) / (np.linalg.norm(raw) + eps)
            ratio = float(np.clip(ratio, 0.1, 10.0))
            update = -lr * ratio * raw
        elif method == "nesterov":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        else:
            update = -lr * grad
        if weight_decay > 0.0 and method not in ["adamw", "lamb"] and key.startswith("W"):
            update -= lr * weight_decay * params[key]
        params[key] += update


def train_mlp(x_tr, y_tr, x_te, y_te, method="sgd", init="he", epochs=12, lr=0.05, hidden=8, batch_size=None, config=None, dropout_p=0.0, dropconnect_p=0.0, seed=0, early_patience=None):
    if config is None:
        config = {}
    classes = int(np.max(y_tr)) + 1
    params = init_weights(x_tr.shape[1], hidden, classes, init, seed)
    state = {}
    rng = np.random.default_rng(seed + 100)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "norm": []}
    best_loss = float("inf")
    best_params = None
    bad_epochs = 0
    n = len(y_tr)
    if batch_size is None:
        batch_size = n
    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            if method == "nesterov":
                lookahead = {}
                for key in params:
                    velocity = state.setdefault("v_" + key, np.zeros_like(params[key]))
                    lookahead[key] = params[key].copy()
                    params[key] += config.get("momentum", 0.9) * velocity
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
                for key in params:
                    params[key] = lookahead[key]
            else:
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
            if method == "kfac":
                grads = kfac_precondition_grads(params, x_tr[idx], y_tr[idx], grads, config.get("damping", 0.03))
                apply_update(params, grads, state, "sgd", lr, epoch, config)
            else:
                apply_update(params, grads, state, method, lr, epoch, config)
        train_loss = eval_loss(params, x_tr, y_tr)
        val_loss = eval_loss(params, x_te, y_te)
        train_acc = accuracy_score(y_tr, predict(params, x_tr))
        val_acc = accuracy_score(y_te, predict(params, x_te))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["norm"].append(vector_norm(params))
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = {key: value.copy() for key, value in params.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if early_patience is not None and bad_epochs >= early_patience:
            params = best_params
            break
    return params, history


def run_component_ladder(variants, metric="accuracy", epochs=12, hidden=16):
    rows = []
    histories = {}
    artifacts = {}
    for rung_index, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        histories[name] = {}
        artifacts[name] = {}
        for variant in variants:
            params, hist = train_mlp(
                x_tr,
                y_tr,
                x_te,
                y_te,
                method=variant.get("method", "sgd"),
                init=variant.get("init", "he"),
                epochs=variant.get("epochs", epochs),
                lr=variant.get("lr", 0.05),
                hidden=hidden,
                batch_size=variant.get("batch_size"),
                config=variant.get("config", {}),
                dropout_p=variant.get("dropout_p", 0.0),
                dropconnect_p=variant.get("dropconnect_p", 0.0),
                seed=variant.get("seed", 10 + rung_index),
                early_patience=variant.get("early_patience"),
            )
            preds = predict(params, x_te)
            acc = accuracy_score(y_te, preds)
            val_loss = eval_loss(params, x_te, y_te)
            value = acc if metric == "accuracy" else val_loss
            rows.append({"rung": name, "variant": variant["name"], "accuracy": acc, "loss": val_loss, "metric": value})
            histories[name][variant["name"]] = hist
            artifacts[name][variant["name"]] = (x_te, y_te, preds)
    return rows, histories, artifacts


def print_table(rows, metric_name):
    print(f"{'rung':34s} {'variant':18s} {metric_name:>10s} {'acc':>8s} {'loss':>8s}")
    for row in rows:
        print(f"{row['rung'][:34]:34s} {row['variant'][:18]:18s} {row['metric']:10.3f} {row['accuracy']:8.3f} {row['loss']:8.3f}")


def plot_results(rows, histories, artifacts, metric_name, best_variant):
    rung_names = list(histories.keys())
    fig, axes = plt.subplots(2, len(rung_names), figsize=(3.2 * len(rung_names), 6.4))
    for col, rung in enumerate(rung_names):
        x_te, y_te, preds = artifacts[rung][best_variant]
        if x_te.shape[1] > 2:
            shown = PCA(n_components=2, random_state=0).fit_transform(x_te)
        else:
            shown = x_te[:, :2]
        axes[0, col].scatter(shown[:, 0], shown[:, 1], c=preds, s=12, cmap="tab10", alpha=0.85)
        axes[0, col].set_title(rung.split("(")[0].strip())
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])
        hist = histories[rung][best_variant]
        curve_key = "val_acc" if metric_name == "accuracy" else "val_loss"
        axes[1, col].plot(hist[curve_key], label=best_variant)
        axes[1, col].set_xlabel("epoch")
        axes[1, col].set_title(metric_name)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    variants = sorted({row["variant"] for row in rows})
    for variant in variants:
        vals = [row["metric"] for row in rows if row["variant"] == variant]
        ax.plot(range(1, len(vals) + 1), vals, marker="o", label=variant)
    ax.set_xticks(range(1, len(rung_names) + 1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(rung_names) + 1)])
    ax.set_ylabel(metric_name)
    ax.set_title("Same ladder, component varied")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## The concept, built once: shrinkage

The lesson formula is
$$\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta g_t.$$
With the lesson's $\eta=0.080$, $g=2.100$, and $\theta=2.000$, the no-decay move is $1.832$.

In [ ]:

def regularizer_sweep(weight_decay, patience):
    theta = 2.0
    eta = 0.080
    gradient = 2.100
    new_theta = (1.0 - eta * weight_decay) * theta - eta * gradient
    return new_theta, patience

no_decay_theta, _ = regularizer_sweep(0.0, 3)
decay_theta, patience = regularizer_sweep(0.1, 3)
print(no_decay_theta, decay_theta, patience)
assert abs(no_decay_theta - 1.832) < 1e-12
assert abs(decay_theta - 1.816) < 1e-12


Early stopping tracks validation loss, saves the best parameter state, and stops after patience is exhausted. It is regularization by limiting training time.

In [ ]:

val_losses = [0.90, 0.70, 0.62, 0.64, 0.67, 0.69]
best_epoch = int(np.argmin(val_losses)) + 1
patience = 2
stop_epoch = best_epoch + patience
print("best epoch", best_epoch, "stop epoch", stop_epoch)
assert best_epoch == 3
assert stop_epoch == 5


## The dataset ladder

Every topic uses the same `clf_digits_ladder()` and the same small MLP. Only the named optimizer or regularization component changes from variant to variant.

In [ ]:

rungs = clf_digits_ladder()
for name, X, y in rungs:
    classes = np.unique(y)
    print(f"{name:38s} shape={X.shape} classes={len(classes)} sample_y={y[:8].tolist()}")
print("D1 sample X:")
print(rungs[0][1])


## Run the same method across D1-D5

The architecture, splits, scaling, and seed policy stay fixed. The table reports one comparable metric per rung.

In [ ]:

variants = [
    {"name": "no decay", "method": "adam", "lr": 0.015},
    {"name": "weight decay", "method": "adamw", "lr": 0.015, "config": {"weight_decay": 0.04}},
    {"name": "early stop", "method": "adamw", "lr": 0.015, "config": {"weight_decay": 0.02}, "early_patience": 4, "epochs": 18},
]

rows, histories, artifacts = run_component_ladder(variants, metric="accuracy", epochs=10, hidden=8)
print_table(rows, "accuracy")


## Results visualization

Top row: small multiples of held-out predictions. Bottom row: validation curves for the highlighted variant, followed by the component summary curve.

In [ ]:

plot_results(rows, histories, artifacts, "accuracy", "early stop")


## Pitfall on D5: train loss can improve while validation loss worsens

Without a validation rule, training can keep optimizing noise. The fix is to retain the best validation epoch and combine it with modest decay.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X, y)
_, long_hist = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", lr=0.02, epochs=18, seed=121)
_, stop_hist = train_mlp(x_tr, y_tr, x_te, y_te, method="adamw", lr=0.015, epochs=18, config={"weight_decay": 0.02}, early_patience=4, seed=121)
best_long = int(np.argmin(long_hist["val_loss"])) + 1
print("long final train/val loss", round(long_hist["train_loss"][-1], 3), round(long_hist["val_loss"][-1], 3))
print("best long validation epoch", best_long)
print("early-stop epochs run", len(stop_hist["val_loss"]))
assert len(stop_hist["val_loss"]) <= 18
assert best_long <= 18


## Evaluate it + Practice

- Main metric: held-out accuracy on every D1-D5 rung, compared with a no-skill baseline near random guessing.
- Sanity check: D1 XOR should improve above chance once the hidden ReLU layer is active.
- Ablation: remove weight decay and disable early stopping for a longer run; the metric should drop or the curve should become less stable.
- Failure signals: exploding loss, flat accuracy near chance, or a D5 train/validation gap that moves in opposite directions.
- Reproducibility: seeds are fixed and the ladder uses sklearn-bundled data only.

Practice 1: Change one hyperparameter in the strongest variant and rerun the summary curve.

Practice 2: Add a new diagnostic printout that distinguishes train accuracy from validation accuracy.

Practice 3: Explain why D5 is harder than D1 using the table and one plotted curve.